# Phase 7 — Stage C: hard top-1 fine-tuning

Stage C initializes from the completed Stage B checkpoint, locks routing to one expert, and fine-tunes with the adaptive entropy gate. It writes atomic resumable checkpoints, per-task loss/PSNR/SSIM matrices, expert-load and entropy statistics, coherence measurements, and trainer status to Google Drive. Rerunning the training cell resumes from this Stage C output directory. DIV2K is stored in Drive so it remains available after a Colab runtime ends. No prediction/intermediate images are written.

The acquisition cell prepares DIV2K, Flickr2K, CBSD68, SOTS, and Outdoor-Rain under Drive, reusing existing folders. DIV2K use is restricted to academic research. Flickr2K's EDSR archive and the CBSD68 community mirror (`https://github.com/clausmichele/CBSD68-dataset`) have public-source download attempts; if unavailable, supply a Drive archive/folder. SOTS is distributed from the [RESIDE dataset page](https://sites.google.com/view/reside-dehaze-datasets/reside-v0); access/download packaging varies. Outdoor-Rain does not identify a unique source in the roadmap, so select the intended corpus and provide an authorized direct URL or Drive archive/file ID. Urban100 acquisition from the EDSR archive is optional for the later benchmark task. Real paired rows can be supplied with `image_path,low_quality_path,task` CSV manifests.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/SparseDict')
DATA_ROOT = Path('/content/drive/MyDrive/SparseDict/datasets')
RUN_DIR = Path('/content/drive/MyDrive/SparseDict/phase7/stage_c')
STAGE_B = Path('/content/drive/MyDrive/SparseDict/phase7/stage_b/latest.pt')
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'master', 'https://github.com/kalpeshjp89/SparseDict.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'master'], check=True)
sys.path.insert(0, str(REPO))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before training.')

In [ ]:
DATASETS = ['div2k', 'flickr2k', 'cbsd68', 'sots', 'outdoor-rain']
DOWNLOAD_PUBLIC_DATASETS = True
DOWNLOAD_URBAN100_FOR_BENCHMARK = False
# DIV2K is academic-research-only; explicitly accept its terms to download.
ACCEPT_DIV2K_TERMS = False
# SOTS/Outdoor-Rain: supply authorized archive paths, Drive IDs, or direct
# HTTPS URLs. Alternatively put extracted data under DATA_ROOT/<dataset>/.
DRIVE_ARCHIVES = {}  # e.g. {'sots': '/content/drive/MyDrive/datasets/SOTS.zip'}
DRIVE_FILE_IDS = {}  # e.g. {'sots': 'Google-Drive-file-id'}
SOURCE_URLS = {}     # e.g. {'outdoor-rain': 'https://authorized-host/data.zip'}
if DOWNLOAD_URBAN100_FOR_BENCHMARK:
    DATASETS.append('urban100')
command = [sys.executable, str(REPO / 'scripts' / 'prepare_phase7_datasets.py'),
           '--root', str(DATA_ROOT)]
for dataset in DATASETS:
    command.extend(['--dataset', dataset])
if DOWNLOAD_PUBLIC_DATASETS:
    command.append('--download-public')
if ACCEPT_DIV2K_TERMS:
    command.append('--accept-div2k-license')
for dataset, path in DRIVE_ARCHIVES.items():
    command.extend(['--drive-archive', f'{dataset}={path}'])
for dataset, file_id in DRIVE_FILE_IDS.items():
    command.extend(['--drive-file-id', f'{dataset}={file_id}'])
for dataset, url in SOURCE_URLS.items():
    command.extend(['--source-url', f'{dataset}={url}'])
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
from scripts.prepare_phase7_datasets import resolve_clean_image_dir
TRAIN_DATASETS = [name for name in DATASETS if name != 'urban100']
# Override a detected folder if a dataset uses a different clean/GT layout.
DATASET_PATH_OVERRIDES = {}  # e.g. {'sots': DATA_ROOT / 'SOTS' / 'outdoor' / 'GT'}
DATASET_DIRS = {
    name: Path(DATASET_PATH_OVERRIDES[name]) if name in DATASET_PATH_OVERRIDES
    else resolve_clean_image_dir(name, DATA_ROOT)
    for name in TRAIN_DATASETS
}
IMAGE_DIRS = list(DATASET_DIRS.values())
if 'urban100' in DATASETS:
    URBAN100_DIR = resolve_clean_image_dir('urban100', DATA_ROOT)
    print('Urban100 benchmark images:', URBAN100_DIR)
print('Fine-tuning image folders:', DATASET_DIRS)
PAIRED_MANIFESTS = []
# Example: PAIRED_MANIFESTS.append(Path('/content/drive/MyDrive/datasets/paired_train.csv'))
if not all(path.is_dir() for path in IMAGE_DIRS):
    raise FileNotFoundError('Download DIV2K or configure IMAGE_DIRS first.')

TRAIN_MANIFEST = RUN_DIR / 'train.csv'
VALIDATION_MANIFEST = RUN_DIR / 'validation.csv'
TASKS = 'sr2,sr3,sr4,denoise15,denoise25,denoise50,derain,desnow,dehaze,lowlight,jpeg'
manifest_command = [
    sys.executable, str(REPO / 'scripts' / 'make_universal_manifest.py'),
    '--train-output', str(TRAIN_MANIFEST), '--validation-output', str(VALIDATION_MANIFEST),
    '--validation-fraction', '0.1', '--seed', '2027', '--tasks', TASKS
]
for directory in IMAGE_DIRS:
    manifest_command.extend(['--image-dir', str(directory)])
for paired_manifest in PAIRED_MANIFESTS:
    manifest_command.extend(['--paired-manifest', str(paired_manifest)])
subprocess.run(manifest_command, cwd=REPO, check=True)
if not STAGE_B.is_file():
    raise FileNotFoundError(f'Run Stage B first: {STAGE_B}')

In [ ]:
command = [
    sys.executable, str(REPO / 'experiments' / 'train_universal.py'),
    '--phase', 'fine_tune', '--train-manifest', str(TRAIN_MANIFEST),
    '--validation-manifest', str(VALIDATION_MANIFEST), '--init-checkpoint', str(STAGE_B),
    '--output-dir', str(RUN_DIR), '--epochs', '20', '--warmup-epochs', '0',
    '--steps-per-epoch', '100', '--patch-size', '128',
    '--validation-samples-per-task', '8',
    '--checkpoint-every-steps', '100', '--device', 'cuda'
]
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('Metrics and status:', RUN_DIR / 'metrics.jsonl', RUN_DIR / 'metrics_by_task.csv', RUN_DIR / 'trainer_status.json')